# Load Data

In [1]:
import pandas as pd
import numpy as np
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import os
from pathlib import Path

# Extract credential access Spotify API

In [2]:
env_path = Path("..") / ".env"
load_dotenv(dotenv_path=env_path)
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
REDIRECT_URI = os.getenv("REDIRECT_URI")

# Extract Spotify data

## Helper Functions

In [3]:
def extract_top_tracks_data(spotify_client,time_range)->pd.DataFrame:
    '''
    Convert JSON file extracted from spotify 
    to dataframe for analysis
    '''
    top_tracks_json = spotify_client.current_user_top_tracks(limit=50, time_range=time_range)
    track_list = []

    for item in top_tracks_json['items']:
        track_info = {
            'track_name':item['name'],
            'artist_name': item['artists'][0]['name'],
            'album_name': item['album']['name'],
            'album_id': item['album']['id'],
            'release_date': item['album']['release_date'],
            'popularity': item['popularity'],
            'duration_ms': item['duration_ms'],
            'explicit':item['explicit'],
            'track_id': item['id']
        }
        track_list.append(track_info)
    
    data = pd.DataFrame(track_list)
    data['duration_min'] = data['duration_ms']/60000
    data['release_date'] = pd.to_datetime(data['release_date'])

    return data

def extract_top_artist_data(spotify_client,time_range)->pd.DataFrame:
    '''
    Convert JSON file extracted from Spotify
    to dataframe for analysis
    '''
    top_artist_json = spotify_client.current_user_top_artists(limit = 50,time_range = time_range)
    artist_list = []
    for item in top_artist_json['items']:
        artist_info = {
            'artist_name':item['name'],
            'popularity':item['popularity'],
            'genres': ', '.join(item['genres']),
            'artist_id': item['id'],
            'followers': item['followers']['total'],
            # Gets the largest image if exists
            'image_url': item['images'][0]['url'] if item['images'] else None
        }
        artist_list.append(artist_info)
    data = pd.DataFrame(artist_list)
    return data

def get_albums_info(spotify_client,album_ids)->pd.DataFrame:
    '''
    Extract album information from top songs 
    '''
    album_data = [] 

    for i in range(0,len(album_ids),20):
        # Spotify API limit requests to 20 albums per call 
        batch = album_ids[i:i+20]
        albums = spotify_client.albums(batch)['albums']
        for album in albums:
            album_info = {
        'album_id': album['id'],
        'album_name': album['name'],
        'release_date': album['release_date'],
        'total_tracks': album['total_tracks'],
        'label': album.get('label', ''),
        'popularity': album.get('popularity'),
        'album_type': album['album_type'],
        'external_url': album['external_urls']['spotify'],
        'album_uri': album['uri'],
        'release_date_precision': album['release_date_precision'],
        'images': [image['url'] for image in album['images']] if album['images'] else [],  # List of images with varying sizes
        'genres': ', '.join(album['genres']) if 'genres' in album else '',  # Check if genres are present
        'artists': ', '.join([artist['name'] for artist in album['artists']]),  # Add artist names
        'track_ids': [track['id'] for track in album['tracks']['items']],  # Extract track IDs
        'available_markets': album.get('available_markets', []),  # Get available markets for the album
        }
            album_data.append(album_info)
    data = pd.DataFrame(album_data)
    return data

def get_tracks_info(spotify_client,track_ids)->pd.DataFrame:
    '''
    Extract additional information on top tracks
    '''
    track_data = [] 

    for i in range(0,len(track_ids),50):
        batch = track_ids[i:i+50]
        tracks = spotify_client.tracks(batch)['tracks']
        for track in tracks:
            track_info = {
                'track_id':track['id'],
                'track_name':track['name'],
                'popularity':track['popularity'],
                'explicit':track['explicit'],
                'artist_id':track['artists'][0]['id'],
                'album_id':track['album']['id'],
                # URL provide 30s preview of track
                'preview_url':track.get('preview_url',''),
                # URL of track page in spotify 
                'track_url':track['external_urls']['spotify'],
                # External Identifier to track songs globally
                'track_external_id':track['external_ids'].get('isrc',''),
                # Check whether track saved in user device
                'is_local':track['is_local'],
            }
            track_data.append(track_info)

    track_data = pd.DataFrame(track_data)
    return track_data

# Fetch Top Tracks

In [4]:
# Optional: remove existing cache for fresh login
if os.path.exists(".cache-my-music-app"):
    os.remove(".cache-my-music-app")

# Create the OAuth object
sp_oauth = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope=["user-top-read", "user-library-read", "playlist-read-private"],
    cache_path=".cache-my-music-app"
)

# Create the Spotify client using the SpotifyOAuth instance directly
sp = spotipy.Spotify(auth_manager=sp_oauth)

# Test API call
try:
    results = sp.current_user_top_tracks(limit=10)
    for idx, item in enumerate(results['items']):
        print(f"{idx+1}. {item['name']} - {item['artists'][0]['name']}")
except spotipy.exceptions.SpotifyException as e:
    print(f"Error: {e}")

print("Fetching your top tracks...")
top_tracks = sp.current_user_top_tracks(
    limit=50, time_range='medium_term'
    )

short_term_track_data = extract_top_tracks_data(sp,'short_term')
medium_term_track_data = extract_top_tracks_data(sp,'medium_term')
long_term_track_data = extract_top_tracks_data(sp,'long_term')

1. CASANOVA POSSE - ALI
2. HER - MINNIE
3. Imaginary Friend - ITZY
4. Air - YEJI
5. I TRUST YOU - エミリア(CV:高橋李依)
6. Shopper - IU
7. Secret - IU
8. Whiplash - aespa
9. FREAK - YUQI
10. NEMONEMO - YENA
Fetching your top tracks...


# Fetch Top Artist

In [5]:
short_term_artists = extract_top_artist_data(sp,'short_term')
medium_term_artists = extract_top_artist_data(sp,'medium_term')
long_term_artists = extract_top_artist_data(sp,'long_term')
long_term_artists

,artist_name,popularity,genres,artist_id,followers,image_url
0,TWICE,79,k-pop,7n2Ycct7Beij7Dj7meI4X0,21665446,https://i.scdn.co/image/ab6761610000e5ebca6c14...
1,(G)I-DLE,73,k-pop,2AfmfGFbe0A0WsTYm0SDTx,10489801,https://i.scdn.co/image/ab6761610000e5eb7fd163...
2,Ed Sheeran,89,soft pop,6eUKZXaKkcviH0Ku9w2n3V,119923718,https://i.scdn.co/image/ab6761610000e5eb399444...
3,Taylor Swift,98,,06HL4z0CvFAxyc27GXpf02,135420613,https://i.scdn.co/image/ab6761610000e5ebe672b5...
4,IU,70,"k-pop, k-ballad",3HqSLMAZ3g3d5poNaI7GOU,9135223,https://i.scdn.co/image/ab6761610000e5ebbd0642...
5,ITZY,69,k-pop,2KC9Qb60EaY0kW4eH68vr3,8351660,https://i.scdn.co/image/ab6761610000e5eb344806...
6,aespa,80,k-pop,6YVMFz59CuY7ngCxTxjpxE,8361617,https://i.scdn.co/image/ab6761610000e5ebf7a109...
7,Against The Current,64,pop punk,6yhD1KjhLxIETFF7vIRf8B,528949,https://i.scdn.co/image/ab6761610000e5eb469aba...
8,MINNIE,63,k-pop,2pHkxVNynHBwQHhGaoBIXX,761745,https://i.scdn.co/image/ab6761610000e5eb1a9a17...
9,IVE,75,k-pop,6RHTUrRF63xao58xh9FXYJ,5451531,https://i.scdn.co/image/ab6761610000e5eb538dad...


# Extract additional info on Tracks

In [6]:
short_term_track_id  = short_term_track_data['track_id'].unique().tolist()
medium_term_track_id  = medium_term_track_data['track_id'].unique().tolist()
long_term_track_id  = long_term_track_data['track_id'].unique().tolist()

short_term_tracks = get_tracks_info(sp,short_term_track_id)
medium_term_tracks = get_tracks_info(sp,medium_term_track_id)
long_term_tracks = get_tracks_info(sp,long_term_track_id)

In [9]:
long_term_tracks

,track_id,track_name,popularity,explicit,artist_id,album_id,preview_url,track_url,track_external_id,is_local
0,47N81NMkB488fuOwOC3Oip,Nobody - from Kaiju No. 8,70,False,5Pwc4xIPtQLFEnJriah9YJ,3YmKf1haPAblZIrIPpuRTf,None,https://open.spotify.com/track/47N81NMkB488fuO...,USUM72403147,False
1,35dhwUoJNlxrPyEIJkfDnx,I GOT YOU,59,False,7n2Ycct7Beij7Dj7meI4X0,6RZHj6L3NqrvcKeiBHQbjL,None,https://open.spotify.com/track/35dhwUoJNlxrPyE...,US5TA2300237,False
2,6ERs9uORCo1MfV0m9ixCuv,FREAK,57,False,22aCD8IrQZjcPgZw728QT6,7LYc8ngbhwha4aGJ5kVauc,None,https://open.spotify.com/track/6ERs9uORCo1MfV0...,KRA392400004,False
3,5vK3WrTOp6rEoASx1jAsp1,DIVE,57,False,7n2Ycct7Beij7Dj7meI4X0,0riep5s1F9ynpobjOSzbcr,None,https://open.spotify.com/track/5vK3WrTOp6rEoAS...,JPWP02470699,False
4,2vNPGH1x5ZwxTjlvzLCyc2,Fate,62,False,2AfmfGFbe0A0WsTYm0SDTx,0mC9MXPddkzggVsOXh5gd3,None,https://open.spotify.com/track/2vNPGH1x5ZwxTjl...,KRA392300030,False
5,65rmgd5uMb4Rgqb5dSiU0p,Doughnut,53,False,7n2Ycct7Beij7Dj7meI4X0,1nqz3cEjuvCMo8RHLBI9kM,None,https://open.spotify.com/track/65rmgd5uMb4Rgqb...,JPWP02170915,False
6,3omvXShuRPM3zbDpWYqf5g,MORE & MORE,63,False,7n2Ycct7Beij7Dj7meI4X0,5KsduuDNWzt65TaHzmtciv,None,https://open.spotify.com/track/3omvXShuRPM3zbD...,US5TA2000050,False
7,4TQBHR8LcbBUv0LvLmn54H,Red Rover,48,False,22aCD8IrQZjcPgZw728QT6,7LYc8ngbhwha4aGJ5kVauc,None,https://open.spotify.com/track/4TQBHR8LcbBUv0L...,KRA392400007,False
8,3Jl2LQmRwbXEF2lO1RTvxn,Full Moon Full Life,65,False,4VeqFgWkP7P9eEGwzPuXcM,20Bf2RVERC5Bc2eo3vyvJv,None,https://open.spotify.com/track/3Jl2LQmRwbXEF2l...,JPK652300101,False
9,26OVhEqFDQH0Ij77QtmGP9,YES or YES,68,False,7n2Ycct7Beij7Dj7meI4X0,25VunQEW0x2W6ALND2Mh4g,None,https://open.spotify.com/track/26OVhEqFDQH0Ij7...,US5TA1800109,False


# Fetch Top Albums

In [7]:
short_term_track_album_id  = short_term_track_data['album_id'].unique().tolist()
medium_term_track_album_id  = medium_term_track_data['album_id'].unique().tolist()
long_term_track_album_id  = long_term_track_data['album_id'].unique().tolist()

short_term_albums = get_albums_info(sp,short_term_track_album_id)
medium_term_albums = get_albums_info(sp,medium_term_track_album_id)
long_term_albums = get_albums_info(sp,long_term_track_album_id)

In [8]:
long_term_albums

,album_id,album_name,release_date,total_tracks,label,popularity,album_type,external_url,album_uri,release_date_precision,images,genres,artists,track_ids,available_markets
0,3YmKf1haPAblZIrIPpuRTf,Nobody (from Kaiju No. 8),2024-04-12,1,Mosley Music/Interscope Records,58,single,https://open.spotify.com/album/3YmKf1haPAblZIr...,spotify:album:3YmKf1haPAblZIrIPpuRTf,day,[https://i.scdn.co/image/ab67616d0000b2732b386...,,OneRepublic,[47N81NMkB488fuOwOC3Oip],"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
1,6RZHj6L3NqrvcKeiBHQbjL,I GOT YOU,2024-02-02,2,Republic Records - TWICE,48,single,https://open.spotify.com/album/6RZHj6L3NqrvcKe...,spotify:album:6RZHj6L3NqrvcKeiBHQbjL,day,[https://i.scdn.co/image/ab67616d0000b273d6a44...,,TWICE,"[35dhwUoJNlxrPyEIJkfDnx, 2iMtwujjXUGjpUraeSrvIm]","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
2,7LYc8ngbhwha4aGJ5kVauc,YUQ1,2024-04-23,7,Cube Entertainment,51,album,https://open.spotify.com/album/7LYc8ngbhwha4aG...,spotify:album:7LYc8ngbhwha4aGJ5kVauc,day,[https://i.scdn.co/image/ab67616d0000b2736f99c...,,YUQI,"[6JgEbE6tFhcKILTL4Z82VA, 6kRV7e933KQw0BoXBkSoA...",[]
3,0riep5s1F9ynpobjOSzbcr,DIVE,2024-07-10,1,WM Japan,46,single,https://open.spotify.com/album/0riep5s1F9ynpob...,spotify:album:0riep5s1F9ynpobjOSzbcr,day,[https://i.scdn.co/image/ab67616d0000b27303661...,,TWICE,[5vK3WrTOp6rEoASx1jAsp1],"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
4,0mC9MXPddkzggVsOXh5gd3,2,2024-01-29,8,Cube Entertainment,60,album,https://open.spotify.com/album/0mC9MXPddkzggVs...,spotify:album:0mC9MXPddkzggVsOXh5gd3,day,[https://i.scdn.co/image/ab67616d0000b27342281...,,(G)I-DLE,"[0uLcuydgTo4ErT6aQQayuw, 3GwmjxMoBSFbcYVvZooKO...",[]
5,1nqz3cEjuvCMo8RHLBI9kM,Celebrate,2022-07-27,9,WM Japan,50,album,https://open.spotify.com/album/1nqz3cEjuvCMo8R...,spotify:album:1nqz3cEjuvCMo8RHLBI9kM,day,[https://i.scdn.co/image/ab67616d0000b27396f40...,,TWICE,"[4Y0chGCyYIRpdUqHJjndF7, 2Ij1NaPpsFSKJ0M65tSV4...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
6,5KsduuDNWzt65TaHzmtciv,MORE & MORE,2020-06-01,7,Republic Records - TWICE,54,album,https://open.spotify.com/album/5KsduuDNWzt65Ta...,spotify:album:5KsduuDNWzt65TaHzmtciv,day,[https://i.scdn.co/image/ab67616d0000b27324869...,,TWICE,"[3omvXShuRPM3zbDpWYqf5g, 128rj96Z6tTEU3h3awSMd...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
7,20Bf2RVERC5Bc2eo3vyvJv,Persona 3 Reload Original Soundtrack,2024-04-24,62,ATLUS GAME MUSIC,71,album,https://open.spotify.com/album/20Bf2RVERC5Bc2e...,spotify:album:20Bf2RVERC5Bc2eo3vyvJv,day,[https://i.scdn.co/image/ab67616d0000b273e59bd...,,"アトラスサウンドチーム, ATLUS GAME MUSIC","[3Jl2LQmRwbXEF2lO1RTvxn, 1d5r3NM4fqH8po53LkMzB...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
8,25VunQEW0x2W6ALND2Mh4g,YES or YES,2018-11-05,7,Republic Records - TWICE,58,album,https://open.spotify.com/album/25VunQEW0x2W6AL...,spotify:album:25VunQEW0x2W6ALND2Mh4g,day,[https://i.scdn.co/image/ab67616d0000b273140ba...,,TWICE,"[26OVhEqFDQH0Ij77QtmGP9, 7fEMfYZnjQ28Cpzi7QnkA...","[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
9,6bj1lsHiVqvJyqjv7qtrV4,"""good guy""",2023-04-21,1,Against the Current,36,single,https://open.spotify.com/album/6bj1lsHiVqvJyqj...,spotify:album:6bj1lsHiVqvJyqjv7qtrV4,day,[https://i.scdn.co/image/ab67616d0000b2737d61e...,,Against The Current,[3bfElZNhtvtGvWMVgCBgZK],"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C..."
